# 01 — Data Exploration
### Landslide Risk Monitoring | SIH 26001 | NER

> **⚠️ DEMO DATA NOTICE:** The dataset used here is **synthetically generated** for MVP/hackathon demonstration only.  
> It does **NOT** represent real NER historical landslide records.  
> Real deployment requires data from IMD, ISRO, GSI, and NDMA.

In [ ]:
# Standard imports
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
import seaborn as sns

# Ensure repo root is on path
REPO_ROOT = Path().resolve().parents[1]  # ml/notebooks/ -> repo root
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

# Plotting style
sns.set_theme(style='darkgrid', palette='viridis')
plt.rcParams['figure.figsize'] = (12, 5)
plt.rcParams['figure.dpi'] = 100

print('✓ Imports OK | Repo root:', REPO_ROOT)

## 1. Generate & Load Dataset

In [ ]:
from ml.datasets.generate_sample import generate_sample_dataset

SAMPLE_PATH = REPO_ROOT / 'ml' / 'datasets' / 'sample' / 'landslide_sample.csv'

# Generate if not already present
if not SAMPLE_PATH.exists():
    print('Generating synthetic sample dataset...')
    generate_sample_dataset(n_samples=1000)

df = pd.read_csv(SAMPLE_PATH)
print(f'Dataset shape: {df.shape}')
df.head()

## 2. Dataset Overview

In [ ]:
print('=== Dataset Info ===')
df.info()
print('\n=== Descriptive Statistics ===')
df.describe().round(2)

## 3. Missing Values

In [ ]:
missing = df.isnull().sum().sort_values(ascending=False)
missing_pct = (missing / len(df) * 100).round(2)
missing_df = pd.DataFrame({'Missing Count': missing, 'Missing %': missing_pct})
print(missing_df[missing_df['Missing Count'] > 0])

# Visual
fig, ax = plt.subplots(figsize=(10, 4))
missing_pct[missing_pct > 0].plot(kind='bar', ax=ax, color='#e74c3c')
ax.set_title('Missing Value Percentage by Column', fontsize=14, fontweight='bold')
ax.set_ylabel('Missing %')
ax.yaxis.set_major_formatter(mtick.PercentFormatter())
plt.tight_layout()
plt.show()

## 4. Target Variable Distribution

In [ ]:
target_counts = df['landslide_occurred'].value_counts()
target_pct = df['landslide_occurred'].value_counts(normalize=True) * 100
print('Class distribution:')
print(pd.DataFrame({'Count': target_counts, 'Percent': target_pct.round(1)}))

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
colors = ['#2ecc71', '#e74c3c']

target_counts.plot(kind='bar', ax=axes[0], color=colors)
axes[0].set_title('Class Count', fontweight='bold')
axes[0].set_xticklabels(['No Landslide (0)', 'Landslide (1)'], rotation=0)
axes[0].set_ylabel('Count')

target_pct.plot(kind='pie', ax=axes[1], colors=colors,
                labels=['No Landslide', 'Landslide'],
                autopct='%1.1f%%', startangle=90)
axes[1].set_title('Class Proportion', fontweight='bold')
axes[1].set_ylabel('')

plt.tight_layout()
plt.show()

## 5. Feature Distributions

In [ ]:
feature_cols = ['rainfall_mm', 'soil_moisture', 'slope_degree',
                'elevation_m', 'terrain_roughness', 'historical_landslide_count']

fig, axes = plt.subplots(2, 3, figsize=(15, 8))
axes = axes.flatten()

for i, col in enumerate(feature_cols):
    ax = axes[i]
    # Plot by class
    for cls, color, label in [(0, '#2ecc71', 'No Landslide'), (1, '#e74c3c', 'Landslide')]:
        subset = df[df['landslide_occurred'] == cls][col].dropna()
        ax.hist(subset, bins=30, alpha=0.6, color=color, label=label, density=True)
    ax.set_title(col, fontweight='bold')
    ax.set_ylabel('Density')
    ax.legend(fontsize=8)

plt.suptitle('Feature Distributions by Class', fontsize=14, fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()

## 6. Correlation Heatmap

In [ ]:
numeric_df = df[feature_cols + ['landslide_occurred']].dropna()
corr = numeric_df.corr()

plt.figure(figsize=(10, 8))
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(
    corr, annot=True, fmt='.2f', cmap='RdYlGn',
    mask=mask, vmin=-1, vmax=1,
    square=True, linewidths=0.5,
)
plt.title('Feature Correlation Matrix', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

# Correlation with target
print('\nCorrelation with target (landslide_occurred):')
target_corr = corr['landslide_occurred'].drop('landslide_occurred').sort_values(ascending=False)
print(target_corr.round(3))

## 7. Geographic Scatter (NER Locations)

In [ ]:
fig, ax = plt.subplots(figsize=(12, 6))

no_slide = df[df['landslide_occurred'] == 0]
slide    = df[df['landslide_occurred'] == 1]

ax.scatter(no_slide['longitude'], no_slide['latitude'],
           c='#2ecc71', alpha=0.4, s=15, label='No Landslide')
ax.scatter(slide['longitude'], slide['latitude'],
           c='#e74c3c', alpha=0.6, s=25, label='Landslide', zorder=5)

ax.set_xlabel('Longitude')
ax.set_ylabel('Latitude')
ax.set_title('Synthetic Location Distribution — NER Bounding Box\n(DEMO DATA — NOT real NER landslide records)',
             fontweight='bold')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 8. Box Plots — Features vs Target

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(15, 8))
axes = axes.flatten()

for i, col in enumerate(feature_cols):
    ax = axes[i]
    df.boxplot(column=col, by='landslide_occurred', ax=ax,
               boxprops=dict(color='#3498db'),
               medianprops=dict(color='#e74c3c', linewidth=2))
    ax.set_title(col, fontweight='bold')
    ax.set_xlabel('Landslide Occurred')
    ax.set_xticklabels(['No (0)', 'Yes (1)'])

plt.suptitle('Feature Distribution by Landslide Class', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

## 9. Summary & Key Observations

In [ ]:
print('=== DATA EXPLORATION SUMMARY ===')
print(f'Total records        : {len(df)}')
print(f'Total features       : {len(feature_cols)}')
print(f'Missing values       : {df.isnull().sum().sum()}')
print(f'Duplicate rows       : {df.duplicated().sum()}')
print(f'Positive class rate  : {df["landslide_occurred"].mean():.1%}')
print()
print('Top correlated features with landslide_occurred:')
print(target_corr.round(3))
print()
print('Next step: Run 02_data_cleaning.ipynb')